# Messkampagne auswerten – Übersicht, Ersatzquellenverfahren, Berichtstabellen

Arbeitet ausschließlich mit den **Ergebnisdateien** der Einzelauswertungen (`Messung_auswerten.ipynb`, Abschnitt 7) –
die Rohdaten werden nicht erneut gelesen. Ausgangsfassung: `Vorlagen/Kampagne_auswerten_DEFAULT.ipynb`.

1. **Übersicht und Konsistenz** der Messungen (gleiche Grundschwingung, Spulenströme, Oberschwingungen sinken mit Z).
2. **Ersatzquellenverfahren**: aktiver Zweipol je Harmonische mit gemeinsamem Winkelbezug (U_L1-L2), Normierung der
   Quellschwankung auf den Spannungspegel vor/nach dem Fehler, zwei Schätzverfahren im Vergleich, paarweise Lösung aus
   je zwei Messungen, Prognosefehler, Jackknife und Monte-Carlo-Unsicherheit – und die **gemessene Quelle**
   (U_q = Vorfehlerspannung des Fehlerleiters, Z_q aus den Versuchen), die über die Zusatzimpedanzen der Versuche
   hinaus trägt.
3. **Erdschlussreststrom an Z_E** mit der gemessenen Quelle (oder der geschätzten, `SOURCE`), die andere zum
   Vergleich, Skalierung auf das Quantil einer **PQ-Messung** der Sammelschiene (aus dem Export des PQ-Messgeräts:
   10-min-Werte, größter Leiter, Quantil, Faktor nach Messdauer) und den eigenen Grenzwerten des Reststroms;
   **resistiver Ansatz** (nach dem VDE Impuls zum Erdschluss-Reststrom).
4. **Station** – Erdungs- und Berührungsspannung je Frequenz (direkt und fehlerbedingt), Vergleich mit den
   Erdungsmessungen aus **groundmeas**, **Reduktionsfaktor** (je Harmonische und effektiv) und – mit PEN-Strömen –
   Strom ins **Ortsnetz** und Z_E,Ortsnetz mit Parallelschaltungsprüfung.
5. **Berichtstabellen** und **Abbildungen** – Export nach `2_Auswertung/Kampagne` und `3_Abbildungen/Kampagne`.
6. **groundmeas** – alle Messungen der Kampagne als Erdschlussmessungen eintragen (Trockenlauf, dann auf Wunsch).
7. **Bericht** als Word-Datei (`REPORT`) mit Platzhaltern für die Fotos vor Ort – in `4_Bericht`.

In [ ]:
%matplotlib inline
import json
import sys
from pathlib import Path

LANGUAGE = "de"                # Sprache von Tabellen, Hinweisen und Abbildungen: "de" | "en"
try:
    import groundfault as gf
except ImportError:            # nicht installiert: Quellordner eines Klons verwenden (…/groundfault/src)
    for p in [Path.cwd(), *Path.cwd().parents]:    # ./src, ./groundfault/src, ./*/groundfault/src
        try:
            candidates = [p / "src", p / "groundfault" / "src", *sorted(p.glob("*/groundfault/src"))]
        except OSError:
            candidates = [p / "src", p / "groundfault" / "src"]
        hits = [c for c in candidates if (c / "groundfault" / "__init__.py").is_file()]
        if hits:
            sys.path.insert(0, str(hits[0]))
            break
    import groundfault as gf
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from groundfault import (campaign, corrections, demo, diagnostics, disturbances, document, earthing,
                         equivalent_source, explorer, i18n, interactive, layout, legacy, pq, register, report, rms, sites,
                         transformer)

i18n.set_language(LANGUAGE)
# Arbeitsordner = dieser Ordner oder der erste Ordner darüber mit einem Kampagnenordner („01_Messkampagnen“
# oder „campaigns“); dessen Name gibt die Ordnerstruktur, ohne ihn gilt die Ordnerstruktur von LANGUAGE
WORK_FOLDER, LAYOUT = layout.find_work_folder(Path.cwd(), default=LANGUAGE)
WORK_FOLDER = Path(WORK_FOLDER)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: "%.4g" % x)
report.activate_style()        # Vorschau im Notebook = Schriftbild der exportierten Abbildungen


def show(df):
    """Tabelle mit Spaltennamen und Codes in der Ausgabesprache."""
    display(i18n.localize(df))


print("groundfault %s · Arbeitsordner: %s · Ordnerstruktur: %s · Schrift der Abbildungen: %s"
      % (gf.__version__, WORK_FOLDER, LAYOUT.language, report.used_font()))

## 1 · Konfiguration

In [ ]:
CAMPAIGN = "JJJJ-MM-TT_Ort"
DEMO     = False
# Ergebnisdateien aus Messung_auswerten.ipynb: 2_Auswertung/<Messung>/<Messung>_ergebnis.json
# Reihenfolge = Spalten der Berichtstabellen. Z = Zusatzimpedanz in Ω (None → nicht im Ersatzquellenverfahren)
MEASUREMENTS = [
    dict(name="Messung 1 (UW 2,5 Ω)", measurement="<Messung_2_5>", Z=2.5),
    dict(name="Messung 2 (UW 5 Ω)",   measurement="<Messung_5>",   Z=5.0),
    dict(name="Messung 3 (UW 0 Ω)",   measurement="<Messung_0>",   Z=0.0),
    dict(name="Messung 4 (Station)",  measurement="<Station>",     Z=None),
]
# optional je Messung: location="…" (genauer Ortsname in groundmeas, sonst aus der Messorte-Datei bzw. dem Messort),
# description="…" (Text vor der automatischen groundmeas-Beschreibung, englisch), timestamp="JJJJ-MM-TT hh:mm:ss"
# (UTC, nur bei falsch gestellter Geräteuhr)

# ---------------------------------------------------------------- Ersatzquellenverfahren
ORDERS        = "auto"             # "auto" = ungerade Ordnungen mit ≥ 0,5 A in ≥ 2 Messungen, sonst z. B. [3, 5, 7, 11, 13]
NORMALIZATION = "complex"          # "complex" (Betrag + Phase) | "magnitude" (nur Betrag) | None
LEVEL         = "mean"             # Quellenpegel: "pre_fault" (Vorfehler) | "post_fault" (Nachfehler) | "mean" (Mittel)
METHOD        = "equation_error"   # "equation_error" (Gleichungsfehler, linear) | "output_error" (Ausgangsfehler, nichtlinear)
SOURCE        = "measured"         # Quelle des Ergebnisses I_RES(Z_E): "measured" (Vorfehlerspannung) | "fitted" (aus den Strömen)
SOURCE_IMPEDANCE = "median"        # Z_q der gemessenen Quelle: "median" über die Versuche | "smallest" (Versuch ohne Zusatzimpedanz, die übrigen prüfen)
Z_E           = None               # Erdungsimpedanz in Ω, an der I_RES berechnet wird – Wert des eigenen Netzes (None = nicht gesetzt)
Z_E_MAX       = 10.0               # Achsenende im Diagramm I_RES(Z_E)
IG_CHOICE     = "max"              # Grundschwingung: "max" (konservativ) | "median" | Zahl in A

# ---------------------------------------------------------------- Skalierung auf das 95-%-Quantil einer PQ-Messung
# U95 skaliert jede Ordnung des Mit- oder Gegensystems (5, 7, 11, 13, 17 …) mit ihrem eigenen 95-%-Wert – auch im Rest
# außerhalb der Ordnungen des Modells; THD95_PERCENT pauschal mit THD95/THD der Messreihe die Ordnungen ohne eigenen
# Wert (allein: alle). Nullsystem (3, 9, 15 …) und gerade Ordnungen bleiben wie gemessen.
U95 = None                         # je Ordnung in V, z. B. {5: U_5, 7: U_7}
U95_REFERENCE = "LL"               # "LL" = Leiter-Leiter (PQ-Gerät) | "seq" = Sequenzspannung (L-E)
THD95_PERCENT = None               # pauschal: THD-95-%-Quantil in % (L-L), allein oder zusammen mit U95
# Mit PQ_FILES übernimmt Abschnitt 4 U95 (Leiter-Leiter, größte Schleife je 10-min-Intervall – Leiter-Erde, größter
# Leiter bei einem Export ohne Leiter-Leiter-Werte) und THD95_PERCENT (größte
# Schleife) aus dem Export bzw. den Exporten eines PQ-Messgeräts an der Sammelschiene – die Werte oben werden ersetzt.
PQ_FILES      = None               # Export(e) eines PQ-Messgeräts an der Sammelschiene, z. B. ["…/Export.csv"] (relativ zum Kampagnenordner oder absolut)
PQ_TIMEZONE   = None               # Zeitzone der Geräteuhr, z. B. "Europe/Berlin" (None = Zeitstempel wie exportiert)
PQ_QUANTILE   = 0.95               # Quantil der 10-min-Werte

# ---------------------------------------------------------------- Grenzwerte, Zuschläge und Faktoren – selbst festlegen
# groundfault enthält keine Grenzwerte oder Sicherheitsfaktoren: aus der Vorgabe des Netzbetreibers bzw. der geltenden
# Norm übernehmen. Nicht gesetzt (die Werte unten), werden die Teile, die sie brauchen, mit einem Hinweis übersprungen.
LIMITS            = ()      # Grenzwerte des Reststroms in A, z. B. (I_1, I_2): Linien in den Diagrammen I_RES(Z_E)
U_TP              = None    # zulässige Berührungsspannung U_Tp in V (resistiver Ansatz; Reduktionsfaktor: None = aus der Ergebnisdatei)
F_FACTOR          = None    # Faktor F der zulässigen Erdungsspannung F·U_Tp (wie U_TP)
REFERENCE_CURRENT = None    # Bezugsstrom in A für den Reduktionsfaktor (None = größter Grenzwert der Ergebnisdatei)
UNCERTAINTY_I     = 0.0     # Messunsicherheitszuschlag m_I der Ströme für I_RES an Z_E: I·(1 + m_I) (0.0 = keiner)
UNCERTAINTY_U     = 0.0     # … m_U der Spannungen: eine mit gemessener Spannung skalierte Ordnung erhält k_m/(1 − m_U) (0.0 = keiner)
LOO_LIMIT_PCT     = None    # Leave-one-out-Prüfung: zulässige Abweichung in % jedes Versuchs vom Modell der übrigen (None = ohne Bewertung)
LOO_LIMIT_A       = None    # … und in A, für die kleinen Ströme der höheren Ordnungen (innerhalb einer der beiden = erfüllt; None = nicht genutzt)
PQ_SAFETY_FACTOR  = 1.0     # Faktor auf die 95-%-Quantile der PQ-Messung (1.0 = keiner)
PQ_FACTORS        = None    # Faktor auf das Quantil nach Dauer der PQ-Messung {Mindesttage: Faktor} (None = PQ_SAFETY_FACTOR)
R_TYPICAL         = None    # erwarteter Bereich (unten, oben) des Reduktionsfaktors bei 50 Hz, z. B. aus Erfahrung

# ---------------------------------------------------------------- groundmeas (Vergleich Abschnitt 5, Eintrag Abschnitt 7)
GROUNDMEAS_DB = None   # None = wie groundmeas (Umgebungsvariable GROUNDMEAS_DB bzw. ~/.config/groundmeas/config.json)
GROUNDMEAS_EXPORT = False          # True: Messungen eintragen (eine Sicherungskopie vorher); "replace": eigene erneuern
GROUNDMEAS_UNCERTAIN = "mark"      # unsichere fehlerbedingte Werte: "mark" (mit Vermerk) | "omit" (weglassen)
GROUNDMEAS_UNITS = "suitable"      # "suitable": je Größe gut lesbar (mV, mA …; groundmeas ≥ 1.6.1) | "base": V und A

# ---------------------------------------------------------------- Resistiver Ansatz (Daten des eigenen Netzes)
RESISTIVE = dict(U_thd95_V=None,   # Oberschwingungsspannung 95 % in V (None = überspringen)
                 from_ll=True,     # True: Wert ist Leiter-Leiter → /√3
                 R_E_max=None, R1_km=None, R0_km=None,   # größter Erdungswiderstand in Ω; Ω/km bei der dominanten Frequenz
                 r=1.0,            # Reduktionsfaktor; r < 1: im Fehlerkreis wirkt r·R_E
                 I_res_max=None)   # zulässiger Erdschlussreststrom in A (None = F_FACTOR·U_TP/(r·R_E_max))

EXPORT_FORMATS = ("pdf", "svg", "png")

# ---------------------------------------------------------------- Bericht (Word, Abschnitt 8)
# Titelseite und Texte (Schlüssel von groundfault.document.ReportInfo; None = kein Bericht). Namen von Netz, Orten und
# Personen erscheinen nur so, wie sie hier stehen; Grenzwerte, U_Tp, F und Zuschläge sind die dieser Konfiguration.
REPORT = dict(network="",     # Zeile unter dem Titel, z. B. "20-kV-Netzbereich UW …"
              authors=[],     # Namen auf der Titelseite
              specified_by="",  # wer Grenzwerte und Faktoren vorgibt, z. B. der Netzbetreiber ("" = neutral: "Vorgaben")
              expansion=True, # Netzausbau (Abschnitt 3.3): True = 1,25/1,5/2 × heutiger I_CE, dict(I_CE=800.0) = geplanter I_CE in A, False = keiner
              places={})      # Name jedes Messorts im Text, z. B. {"UW": "Umspannwerk …", "<Stationsordner>": "…"}
PHOTOS = {}                        # Fotos vor Ort in <Medien>: {"setup_impedance": …, "recorder": …, "station": …} (fehlt: Platzhalter)
REPORT_TEMPLATE = None             # Word-Datei der eigenen Organisation (Formatvorlagen, Kopf-/Fußzeile, Logo), z. B. mit document.make_template
REPORT_STYLES = None               # Namen der Formatvorlagen der Vorlage, z. B. {"title": "…", "subtitle": "…", "table": "…"}
REPORT_PDF = False                 # zusätzlich ein PDF mit LibreOffice schreiben (falls installiert)

## 2 · Ergebnisse laden – Übersicht und Konsistenz

In [ ]:
CAMPAIGN_FOLDER = WORK_FOLDER / LAYOUT.campaigns / CAMPAIGN
for m in MEASUREMENTS:                 # nur bekannte Schlüssel (ein vertippter oder früherer Schlüssel bliebe unbeachtet)
    legacy.check_keys("MEASUREMENTS", m, ("name", "measurement", "Z", "location", "description", "timestamp"))


def result_file(measurement):
    """Ergebnisdatei einer Messung (auch mit der Endung der anderen Ordnerstruktur)."""
    p = Path(LAYOUT.result_path(CAMPAIGN_FOLDER, measurement))
    other = [p.parent / (measurement + lay.result_suffix) for lay in layout.LAYOUTS.values()]
    return next((x for x in [p, *other] if x.exists()), p)


if DEMO:                               # Demo: Kampagne anlegen und die Aufzeichnungen mit ihren bekannten Bereichen auswerten
    if not CAMPAIGN_FOLDER.exists():
        demo.make_campaign(str(WORK_FOLDER), CAMPAIGN, layout=LAYOUT.language)
    CORR = corrections.load(CAMPAIGN_FOLDER)      # Korrekturen der Kampagne wie in Messung_auswerten.ipynb
    for m in MEASUREMENTS:
        if not result_file(m["measurement"]).exists():
            cfg = next(CAMPAIGN_FOLDER.glob("%s/*/%s.cfg" % (LAYOUT.raw, m["measurement"])))
            with open(str(cfg).replace(".cfg", "_truth.json"), encoding="utf-8") as f:
                b = json.load(f)["intervals"]
            rec_ = gf.open_recording(str(cfg))
            CORR.apply(rec_)
            roles_ = gf.detect_roles(rec_.channel_names, rec_.units, CORR.role_overrides(rec_), CORR.main_channel(rec_),
                                     CORR.distances(rec_))
            r_ = gf.evaluate(rec_, gf.Intervals(*(tuple(b[k]) for k in ("steady_state", "pre_fault", "post_fault"))),
                             gf.Settings(uncertainty_I=UNCERTAINTY_I, uncertainty_U=UNCERTAINTY_U, ires_limits=LIMITS,
                                         u_tp=U_TP, f_factor=F_FACTOR),
                             roles=roles_, z_fault_ohm=m["Z"])
            r_.save_json(LAYOUT.result_path(CAMPAIGN_FOLDER, m["measurement"]))
RESULTS = {}
for m in MEASUREMENTS:
    p = result_file(m["measurement"])
    if p.exists():
        RESULTS[m["name"]] = gf.load_result(str(p), m["name"])
    else:
        print("⚠ fehlt:", p.relative_to(WORK_FOLDER), "→ zuerst mit Messung_auswerten.ipynb auswerten")
show(campaign.overview(RESULTS))
display(Markdown("**Konsistenz zwischen den Messungen**"))
show(campaign.consistency({m["name"]: RESULTS[m["name"]] for m in MEASUREMENTS if m["Z"] is not None and m["name"] in RESULTS}))
for n, e in RESULTS.items():
    for h in e.hints:
        print("⚠ %s: %s" % (n, h))

## 3 · Ersatzquellenverfahren (aktiver Zweipol je Harmonische)

**Normierung:** Der Oberschwingungspegel des Netzes schwankt zwischen den Messungen (Minuten). Jeder Strom wird auf den
gleichzeitig gemessenen Pegel der natürlichen Sequenz *am Fehlerleiter* bezogen (5./11. Gegensystem, 7./13. Mitsystem;
bei L2/L3 um ±120° gedreht). Ordnungen des Nullsystems (3., 9.) werden **nicht** normiert – ihr Strom entsteht erst im
Erdschluss.
**Zwei Schätzer:** *Gleichungsfehler* (lineare Ausgleichsrechnung) und *Ausgangsfehler* (minimiert die
Stromabweichung). Liegen beide weit auseinander, sind U_q und Z_q einzeln unsicher (der Hinweis nennt, ob die
Vorhersage im Messbereich trotzdem stabil ist).
**Thevenin-Kontrolle:** Physikalisch ist U_q die Vorfehlerspannung des Fehlerleiters → Spalte
`Uq_bez_Fehlerleiter` ≈ 1 und `Uq_Winkel_bez_Fehlerleiter_deg` ≈ 0°. Mehr als 10 % bzw. 10° Abweichung zeigen, dass
das geschätzte U_q Änderungen des Netzes zwischen den Versuchen aufgenommen hat: U_q ist die Steigung von 1/I über Z,
schlecht bestimmt, wenn die Zusatzimpedanzen klein gegen |Z_q| sind. Im Bereich der Versuche bleibt der Modellstrom
belastbar, darüber hinaus nicht. **Empfindlichkeit**: relative Änderung von Z_q bei 1 % (bzw. 0,5°) Fehler eines
Stroms – über 0,1 schlecht konditioniert.
**Gemessene Quelle:** U_q wie gemessen – die Vorfehlerspannung des Fehlerleiters – und Z_q = U_q/I_k − Z_k aus den
Versuchen (`SOURCE_IMPEDANCE`: der Median über die Versuche oder der Versuch ohne Zusatzimpedanz); ein Versuch genügt,
jeder weitere wird mit dem Modell der übrigen geprüft (Leave-one-out, `LOO_LIMIT_PCT`). Die Tabelle der einzelnen
Versuche zeigt, wie stark sich das Netz zwischen ihnen geändert hat, der Vergleich, wo beide Quellen übereinstimmen
(Messbereich) und wo nicht. Ordnungen des Nullsystems behalten die
geschätzte Quelle – ihre Quelle entsteht im Fehler. `SOURCE` wählt das Ergebnis der Kontrollen und von Abschnitt 4;
die andere Quelle ist gestrichelt eingezeichnet.

In [ ]:
SERIES = MEASURED = MAIN = OTHER = MC = None   # geschätzte Quelle, gemessene Quelle, Ergebnis (SOURCE), Vergleich; MC: Monte Carlo (Kontrollen)
SOURCE_TEXT = {"measured": "gemessene Quelle", "fitted": "geschätzte Quelle"}
entries = [dict(file=str(result_file(m["measurement"])), Z=m["Z"], name=m["name"])
           for m in MEASUREMENTS if m["Z"] is not None and m["name"] in RESULTS]
if entries:
    legacy.check_option("SOURCE", SOURCE, ("measured", "fitted"))
    legacy.check_option("SOURCE_IMPEDANCE", SOURCE_IMPEDANCE, ("median", "smallest"))
    TESTS = equivalent_source.load_series(entries)
    if len({complex(e["Z"]) for e in entries}) >= 2:
        SERIES = equivalent_source.fit_equivalent_source(TESTS, orders=ORDERS, normalization=NORMALIZATION,
                                                         reference=LEVEL, method=METHOD)
        MEASURED = equivalent_source.measured_source(SERIES, impedance=SOURCE_IMPEDANCE)
    else:                              # eine Zusatzimpedanz: keine geschätzte Quelle, die gemessene genügt
        MEASURED = equivalent_source.measured_source(TESTS, impedance=SOURCE_IMPEDANCE, orders=ORDERS, reference=LEVEL)
        print("ℹ Nur eine Zusatzimpedanz: die geschätzte Quelle braucht zwei – Ergebnis mit der gemessenen Quelle.")
    MAIN, OTHER = (MEASURED, SERIES) if SOURCE == "measured" or SERIES is None else (SERIES, MEASURED)
    show(MAIN.measurements_table())
    display(Markdown("**Quellenpegel vor/nach dem Fehler** (Drift innerhalb einer Messung = Güte der Normierung)"))
    show(MAIN.level_table(LEVEL))
    if SERIES is not None:
        display(Markdown("**Geschätzte Quelle: Zweipol-Parameter je Ordnung** (Thevenin-Kontrolle ≈ 1∠0°)"))
        show(SERIES.source_table())
        for h in SERIES.hints:
            print("ℹ", h)
        for m, F in SERIES.fits.items():
            for h in F.hints:
                print("ℹ %d. Harmonische: %s" % (m, h))
    display(Markdown("**Gemessene Quelle** – U_q = Vorfehlerspannung des Fehlerleiters; Z_q jedes einzelnen Versuchs (die Streuung ist die Änderung des Netzes zwischen den Versuchen) und der Reihe"))
    show(MEASURED.single_test_table())
    show(MEASURED.source_table()[["order", "f_Hz", "sequence", "method", "Uq_V", "Rq_Ohm", "Xq_Ohm", "Zq_Ohm",
                                  "residual_pct", "Rq_fixed"]])
    for h in (MEASURED.hints[-1:] if SERIES is not None else MEASURED.hints):    # mit geschätzter Reihe: nur der neue Hinweis
        print("ℹ", h)
    for m, F in MEASURED.fits.items():
        for h in (F.hints if F.method == "measured" else F.hints[-1:]):     # beibehaltene geschätzte Ordnungen: nur der neue Hinweis
            print("ℹ %d. Harmonische: %s" % (m, h))
    if SERIES is not None:
        display(Markdown("**Gemessene gegen geschätzte Quelle** – Strom je Ordnung und I_RES im Messbereich und darüber hinaus (`geschaetzt_zu_gemessen`; `Interpolation` = im Bereich der Zusatzimpedanzen der Versuche)"))
        zmin, zmax = SERIES.Z_range
        show(equivalent_source.compare_sources(MEASURED, SERIES, [0.0, zmax, 2 * zmax]
                                               + ([Z_E] if Z_E is not None and Z_E not in (0.0, zmax, 2 * zmax) else []),
                                               IG=IG_CHOICE))
else:
    print("Für das Ersatzquellenverfahren ist mindestens eine UW-Messung mit ihrer Zusatzimpedanz Z nötig (für die geschätzte Quelle zwei verschiedene).")

### Kontrollen: paarweise Lösung, Prognose je Messung, Jackknife, Monte Carlo

In [ ]:
if MAIN is not None:
    if SERIES is not None:
        display(Markdown("**Geschätzte Quelle: Lösung aus je zwei Versuchen**"))
        show(SERIES.pairwise_table())
    display(Markdown(i18n.format_text("**Ergebnis ({}): Modell gegen Messung je Versuch, Jackknife an Z_E**",
                                      SOURCE_TEXT[MAIN.settings["source"]])))
    show(MAIN.forecast_table())
    display(Markdown("**Leave-one-out: jeder Versuch aus dem Modell der übrigen** (große Abweichung: Netz oder Quellen haben sich zwischen den Versuchen geändert)"))
    LOO = MAIN.loo_table(LOO_LIMIT_PCT, LOO_LIMIT_A)
    show(LOO)
    BAD = LOO["within_limit"].eq(False) if "within_limit" in LOO else LOO["order"].isna()   # None: nicht bestimmbar
    if BAD.any():
        print(i18n.format_text("⚠ Leave-one-out: {} von {} Werten außerhalb der zulässigen Abweichung ({}) – ein Hinweis auf eine Änderung von Netz oder Quellen zwischen den Versuchen",
                               int(BAD.sum()), int(LOO["within_limit"].notna().sum()),
                               " / ".join(i18n.format_text(f, v) for f, v in (("±{:g} %", LOO_LIMIT_PCT), ("±{:g} A", LOO_LIMIT_A)) if v is not None)))
    if Z_E is not None:
        show(MAIN.jackknife_table(Z_E))
    display(Markdown("**Reststrom an den Messimpedanzen** – Modell (mit Pegel und I_G der jeweiligen Messung, Rest = nicht gefittete Ordnungen) gegen den gemessenen Echt-Effektivwert"))
    show(MAIN.check_table())
    if Z_E is None:
        print("ℹ Jackknife und Monte Carlo an Z_E übersprungen – Z_E ist nicht gesetzt (Abschnitt 1).")
    else:
        MC = equivalent_source.monte_carlo(MAIN, Z_E, n=2000, IG=IG_CHOICE)
        print(i18n.format_text("Monte Carlo (1 %/0,5° Strom, 2 % Z, Modellresiduum): I_RES(Z_E = {:g} Ω) P5 … P95 = {:.1f} … {:.1f} A (Median {:.1f} A)",
                               Z_E, MC["IRES"][5], MC["IRES"][95], MC["IRES"][50]))
        zmin, zmax = MAIN.Z_range
        print(i18n.format_text("Z_E liegt {} des Messbereichs {:.1f} … {:.1f} Ω",
            "innerhalb" if zmin <= abs(Z_E) <= zmax else "AUSSERHALB (Extrapolation)", zmin, zmax))
    interactive.source_fit(MAIN, Z_E, comparison=OTHER).show()

## 4 · Erdschlussreststrom an der Erdungsimpedanz und resistiver Ansatz

### Netzqualitätsmessung der Sammelschiene (PQ)

Mit `PQ_FILES` ersetzt der Export bzw. ersetzen die Exporte eines PQ-Messgeräts an der Sammelschiene `U95`,
`U95_REFERENCE`, `THD95_PERCENT` und `PQ_SAFETY_FACTOR` aus Abschnitt 1: 10-min-Werte (aus kürzeren Intervallen als
quadratischer Mittelwert), je Intervall der **größte Leiter** (Oberschwingungsspannung Leiter-Erde je Ordnung) bzw.
die **größte Schleife** (THD Leiter-Leiter), davon das Quantil `PQ_QUANTILE`. Der Faktor auf das Quantil folgt aus der
längsten Strecke **lückenloser** Werte und `PQ_FACTORS`; kürzer als jede Dauer von `PQ_FACTORS` ist die Messung nicht
gültig, und mit ihr wird nichts skaliert. PQ-Messung und Versuche müssen sich auf dasselbe Netz und denselben
Schaltzustand beziehen. Ohne Wert in `RESISTIVE["U_thd95_V"]` erhält der resistive Ansatz das Quantil der
Oberschwingungsspannung (Leiter-Leiter) mal Faktor. Die Tabelle der Versuche zeigt, wo deren Vorfehlerspannungen in
der Verteilung der PQ-Messung liegen: Ein niedriges Perzentil ist normal für einen Versuch zu einer Zeit kleiner
Oberschwingungsspannung (Nacht, Wochenende) – dazu das Zeitfenster desselben Wochentags vergleichen; ein Pegel über
dem größten Wert der PQ-Messung deutet auf einen anderen Netzzustand (Schaltzustand, Kapazitäten, Trafo). Abbildungen:
10-min-Werte über den Zeitraum, Dauerlinie mit den Pegeln der Versuche und Wochengang.

In [ ]:
PQ = PQ_COMPARISON = PQ_LEVELS = None
if PQ_FILES:
    files = [CAMPAIGN_FOLDER / p for p in ([PQ_FILES] if isinstance(PQ_FILES, (str, Path)) else PQ_FILES)]
    PQ = pq.assess(pq.read(files, timezone=PQ_TIMEZONE), quantile=PQ_QUANTILE,
                   **(dict(factors=PQ_FACTORS) if PQ_FACTORS else dict(factor=PQ_SAFETY_FACTOR)))
    print(PQ.summary())
    for h in PQ.hints:
        print("ℹ", h)
    show(PQ.table().query("order >= 2"))
    show(PQ.summary_table())
    if PQ.valid:                       # die Werte der Messung ersetzen die aus Abschnitt 1
        U95, U95_REFERENCE = PQ.u95(), ("seq" if PQ.reference == "LE" else "LL")
        THD95_PERCENT, PQ_SAFETY_FACTOR = (PQ.thd95() if np.isfinite(PQ.thd95()) else None), PQ.factor
        if RESISTIVE.get("U_thd95_V") is None and np.isfinite(PQ.uh95_ll):
            RESISTIVE = dict(RESISTIVE, U_thd95_V=PQ.uh95_ll * PQ.factor, from_ll=True)
            print(i18n.format_text("Resistiver Ansatz: U_thd95_V = {:.1f} V (Quantil der Oberschwingungsspannung Leiter-Leiter × {:g})",
                                   RESISTIVE["U_thd95_V"], PQ.factor))
    else:
        print("⚠ Die PQ-Messung ist für die Bewertung nicht gültig (lückenlose Werte kürzer als jede Dauer von PQ_FACTORS) – mit ihr wird nichts skaliert; U95/THD95_PERCENT aus Abschnitt 1 bleiben.")
    PQ_LEVELS, times = {}, {}
    for m in MEASUREMENTS:
        if m["Z"] is not None and m["name"] in RESULTS:
            PQ_LEVELS[m["name"]], times[m["name"]] = pq.pre_fault_levels(RESULTS[m["name"]], kind=PQ.reference)
    orders = sorted(o for o in (MAIN.fits if MAIN is not None else PQ.u95()) if gf.formulas.harmonic_category(o) == "pos_neg"
                    and o in PQ.data.orders)
    PQ_COMPARISON = PQ.comparability(PQ_LEVELS, times, orders=orders or None)
    for h in PQ_COMPARISON.attrs.get("hints", []):
        print("ℹ", h)
    display(Markdown("**Erdschlussversuche in der Verteilung der PQ-Messung** – Anteil der 10-min-Werte unter der Vorfehlerspannung des Versuchs (größte Schleife bzw. größter Leiter) über die ganze Messung und im Zeitfenster des Versuchs (nur bei synchronisierten Uhren von Rekorder und PQ-Gerät aussagekräftig); `Faktor_zu_Versuch` = Quantil/Pegel des Versuchs"))
    show(PQ_COMPARISON)
    report.pq_series(PQ)
    report.pq_duration(PQ, levels={n: lv[PQ.dominant_order] for n, lv in PQ_LEVELS.items() if PQ.dominant_order in lv})
    report.pq_profile(PQ)
else:
    print("ℹ Keine PQ-Messung (PQ_FILES) – U95/THD95_PERCENT aus Abschnitt 1, falls gesetzt.")

In [ ]:
SCALING = None
if MAIN is not None:
    if U95 or THD95_PERCENT:           # U95 je Ordnung; THD95_PERCENT für die Ordnungen ohne eigenen Wert (allein: alle)
        SCALING = dict(safety_factor=PQ_SAFETY_FACTOR)
    if U95:
        SCALING.update(u95=U95, u95_reference=U95_REFERENCE)
    if THD95_PERCENT:
        thd_measured = float(pd.concat([e.tables["voltage_metrics"] for n, e in RESULTS.items()
                                        if n in [x["name"] for x in entries]])
                             .query("interval != 'fault'")[["UTHD12_pct", "UTHD23_pct", "UTHD31_pct"]].max(axis=1).mean())
        SCALING.update(thd95_percent=THD95_PERCENT, thd_measured_percent=thd_measured)
        print(i18n.format_text("THD der Messreihe (Mittel vor/nach, max. Schleife): {:.2f} %", thd_measured))
    if Z_E is None:
        print("ℹ Reststrom an Z_E übersprungen – Z_E ist nicht gesetzt (Abschnitt 1); das Diagramm zeigt I_RES über Z_E.")
    else:
        r = equivalent_source.ires_at_impedance(MAIN, Z_E, IG=IG_CHOICE, uncertainty_I=UNCERTAINTY_I,
                                                uncertainty_U=UNCERTAINTY_U, **(SCALING or {}))
        margin = UNCERTAINTY_I > 0 or UNCERTAINTY_U > 0   # den Wert mit Zuschlag nur, wenn einer gesetzt ist
        print(i18n.format_text("{}: I_G = {:.2f} A | I_H(Z_E) = {:.2f} A (davon Rest {:.2f} A) | I_RES(Z_E = {:g} Ω) = {:.2f} A",
                               SOURCE_TEXT[MAIN.settings["source"]], r["IG"], r["IH"], r["IH_rest"], Z_E, r["IRES"])
              + (i18n.format_text(" | mit Messunsicherheitszuschlag: {:.2f} A", r["IRES_assessed"]) if margin else ""))
        if SCALING:
            print(i18n.format_text("skaliert: I_H = {:.2f} A → I_RES = {:.2f} A", r["IH_scaled"], r["IRES_scaled"])
                  + (i18n.format_text(" (mit Zuschlag: {:.2f} A)", r["IRES_scaled_assessed"]) if margin else "")
                  + i18n.format_text(" | nicht skaliert: Nullsystem {:.2f} A, gerade Ordnungen {:.2f} A, ohne Ordnung {:.2f} A",
                                     r["IH_zero"], r["IH_even"], r["IH_other"]))
        display(Markdown(i18n.format_text("**Ergebnis je Ordnung an Z_E = {:g} Ω** – die Ordnungen des Modells an Z_E, der Rest des Versuchs {} (der größte); Regel: `eigene Spannung` (Mit-/Gegensystem, Quantil/Pegel der Versuche × Faktor), `THD` (ohne eigenen Wert), `wie gemessen` (Nullsystem, gerade Ordnungen)",
                                          Z_E, r["rest_test"] or "–")))
        show(r["table"])
        for h in r["hints"]:
            print("ℹ", h)
        if OTHER is not None:
            o = equivalent_source.ires_at_impedance(OTHER, Z_E, IG=IG_CHOICE, **(SCALING or {}))
            print(i18n.format_text("Vergleich {}: I_H(Z_E) = {:.2f} A | I_RES(Z_E) = {:.2f} A",
                                   SOURCE_TEXT[OTHER.settings["source"]], o["IH"], o["IRES"])
                  + (i18n.format_text(" | skaliert: I_RES = {:.2f} A", o["IRES_scaled"]) if SCALING else ""))
    if not LIMITS:
        print("ℹ Keine Grenzwertlinien in den Diagrammen I_RES(Z_E) – LIMITS ist nicht gesetzt (Abschnitt 1).")
    interactive.ires_ze(MAIN, Z_E_MAX, LIMITS, SCALING, comparison=OTHER).show()
legacy.check_keys("RESISTIVE", RESISTIVE, ("U_thd95_V", "from_ll", "R_E_max", "R1_km", "R0_km", "r", "I_res_max"))
NOT_SET = [k for k in ("U_thd95_V", "R_E_max", "R1_km", "R0_km") if RESISTIVE.get(k) is None]
if RESISTIVE.get("I_res_max") is None:                # sonst I_RES,max = F·U_Tp/(r·R_E,max)
    NOT_SET += [n for n, v in (("U_TP", U_TP), ("F_FACTOR", F_FACTOR)) if v is None]
if NOT_SET:
    print("ℹ Resistiver Ansatz übersprungen – nicht gesetzt (Abschnitt 1): %s"
          % ", ".join(n if n in ("U_TP", "F_FACTOR") else "RESISTIVE[%r]" % n for n in NOT_SET))
else:
    IG_ALL = [e.values["IG"] for e in RESULTS.values() if np.isfinite(e.values.get("IG", np.nan))] or [0.0]
    if isinstance(IG_CHOICE, str):
        legacy.check_option("IG_CHOICE", IG_CHOICE, ("max", "median"))
    IG = (max(IG_ALL) if IG_CHOICE == "max" else float(np.median(IG_ALL)) if IG_CHOICE == "median" else float(IG_CHOICE))
    display(pd.Series(campaign.resistive_approach(RESISTIVE["U_thd95_V"], IG, RESISTIVE["R_E_max"], RESISTIVE["R1_km"],
                                                  RESISTIVE["R0_km"], I_res_max=RESISTIVE.get("I_res_max"),
                                                  U_Tp=U_TP, F=F_FACTOR, from_ll=RESISTIVE.get("from_ll", False),
                                                  r=RESISTIVE.get("r", 1.0)),
                      name="resistiver Ansatz"))

### PQ-Skalierung an Z_E: je Ordnung, als Zeitreihe und mit dem THD

**Ergebnis** (`SCALING`): jede Ordnung des Mit- oder Gegensystems (5, 7, 11, 13, 17 …) mit ihrem eigenen Faktor
Quantil/Pegel der Versuche mal Faktor – auch im Rest außerhalb der Ordnungen des Modells; eine Ordnung ohne eigenen
Wert (kein Quantil, Pegel unter 1 V) mit dem THD-Faktor. Das Nullsystem (3, 9, 15 …) entsteht im Erdschluss und bleibt
wie gemessen; die geraden Ordnungen bleiben wie gemessen und werden gesondert aufgeführt (untypisch – ein Hinweis nennt
sie ab 5 % des Oberschwingungsanteils). **Zeitreihe**: dasselbe für jeden 10-min-Wert, davon von I_RES das Quantil
`PQ_QUANTILE` (mal Faktor) – kleiner, wenn die Ordnungen ihre hohen Werte nicht gleichzeitig erreichen. **THD**: ein
pauschaler Faktor THD-Quantil/THD der Versuche für jede Ordnung des Mit- oder Gegensystems – die Regel ohne Werte je
Ordnung.

In [ ]:
PQ_SCALING = None
if PQ is not None and PQ.valid and MAIN is not None and Z_E is not None and SCALING:
    PQ_SCALING = PQ.scaling_methods(MAIN, Z_E, IG=IG_CHOICE, uncertainty_I=UNCERTAINTY_I, uncertainty_U=UNCERTAINTY_U,
                                    scaling=SCALING)
    display(Markdown(i18n.format_text("**Erdschlussreststrom an Z_E = {:g} Ω mit der PQ-Messung** (Quantil {:g}, Faktor {:g}; Bewertung = mit UNCERTAINTY_I und UNCERTAINTY_U)",
                                      Z_E, PQ_QUANTILE, PQ.factor)))
    show(PQ_SCALING)
    for h in PQ_SCALING.attrs.get("hints", []):
        print("ℹ", h)
    for limit in LIMITS:
        over = PQ_SCALING.loc[PQ_SCALING["IRES_assessed_A"] > limit, "method"]
        print(i18n.format_text("I_RES über {:g} A: ", limit) + (", ".join(over) or "–"))
elif PQ is not None and not PQ.valid:
    print("ℹ Keine PQ-Skalierung an Z_E – die PQ-Messung ist für die Bewertung nicht gültig (Abschnitt 4, PQ).")
elif PQ is not None:
    print("ℹ Kein Vergleich an Z_E – Z_E oder die Ersatzquellen fehlen (Abschnitte 1 und 3).")

## 5 · Station: Erdungs-/Berührungsspannung je Frequenz und Erdungsmessung (groundmeas)

Für jede Stationsmessung der Kampagne (Ergebnisdatei mit Tabelle `station_frequency`): Spannung je Ampere
Fehlerstrom über der Frequenz – **direkt** gemessen und **fehlerbedingt** (ohne die Grundlast vor dem Fehler) – und,
falls der Ort in groundmeas erfasst ist, der Vergleich mit der Erdungsmessung (Einspeisung) samt Verhältnis
(für die Erdungsspannung an einer fernen Sonde = wirksamer Reduktionsfaktor). groundmeas wird nur gelesen.
**Reduktionsfaktor** je Harmonische und effektiv (Erdungsspannung je Ampere, Erdschluss ÷ Erdungsmessung, Beträge;
maßgeblich das Sondenmaximum `max(UE)` gegen die fernste Sonde der Erdungsmessung) mit Einordnung (gegen den eigenen
Bereich `R_TYPICAL` bei 50 Hz, fallend mit f) und Bedeutung der Abweichung für die Bewertung (Spannung beim Bezugsstrom
gegen F·U_Tp – `REFERENCE_CURRENT`, `U_TP` und `F_FACTOR` aus Abschnitt 1, sonst die Werte der Ergebnisdatei); mit
PEN-Strömen außerdem der Strom ins **Ortsnetz** und Z_E,Ortsnetz = ΔU_E/I_ON samt Parallelschaltungsprüfung (gegen
den Erdschluss max(U_E)/I_F und den Erdungsstrom) und dem Vergleich mit der Erdungsmessung.

In [ ]:
STATIONS = {n: e for n, e in RESULTS.items() if not e.tables.get("station_frequency", pd.DataFrame()).empty}
if STATIONS and R_TYPICAL is None:
    print("ℹ Keine Einordnung von r bei 50 Hz – R_TYPICAL ist nicht gesetzt (Abschnitt 1).")
COMPARISONS = {}
REDUCTION = {}                                                    # Reduktionsfaktor und Parallelschaltung je Station
DB = None
try:
    SITES = sites.load(CAMPAIGN_FOLDER)                            # Messorte-Datei (Ort in groundmeas)
except ValueError as ex:
    SITES = sites.Sites()
    print("⚠", ex)
if STATIONS:
    try:
        DB = earthing.Database(GROUNDMEAS_DB)
    except (FileNotFoundError, ValueError) as ex:
        print("ℹ groundmeas-Datenbank nicht verfügbar:", ex)
else:
    print("Keine Stationsmessung mit frequenzselektiver Auswertung.")
for n, e in STATIONS.items():
    display(Markdown("### %s" % n))
    show(e.table("station_frequency"))
    interactive.station_frequency(e).show()
    if not e.tables.get("station_lv_network", pd.DataFrame()).empty:
        display(Markdown("**Ortsnetz** – Strom I_ON und Z_E,Ortsnetz = ΔU_E/I_ON (je Frequenz und effektiv)"))
        show(e.table("station_lv_network_eff")); show(e.table("station_lv_network"))
    if DB is None:
        continue
    m = next(x for x in MEASUREMENTS if x["name"] == n)
    site = e.data.get("meta", {}).get("site") or ""
    entry = SITES.for_site(site)
    location = m.get("location") or (entry.location if entry is not None and entry.location else None) or \
        earthing.location_from_site(site or m["measurement"])
    choice = DB.choose_location(location, exact_only=bool(m.get("location") or (entry is not None and entry.location)),
                                preset='location="{}" in MEASUREMENTS')
    if choice["note"]:
        print("ℹ", choice["note"])
    if choice["location"] is None:
        continue
    meas = earthing.exclude_own(DB.measurements(location=choice["location"]), CAMPAIGN, m["measurement"])
    response = earthing.frequency_response(DB, meas["measurement_id"])
    long_df, ratio = earthing.compare(e.table("station_frequency"), response, measurement=n)
    COMPARISONS[n] = (long_df, ratio)
    print("groundmeas: Ort „%s“, Messungen %s" % (choice["location"], ", ".join(map(str, meas["measurement_id"]))))
    if len(ratio):
        show(ratio)
    if long_df["source"].isin([earthing.SOURCE_INJECTION, earthing.SOURCE_GROUNDMEAS_FAULT]).any():
        interactive.earthing_comparison(long_df, ratio).show()
    rf, rf_eff = earthing.reduction_factor(e, response, reference_current_A=REFERENCE_CURRENT, U_Tp=U_TP, F=F_FACTOR,
                                           r_typical=R_TYPICAL)  # None: aus der Ergebnisdatei
    not_set = [v for v, c in (("REFERENCE_CURRENT", "reference_current_A"), ("U_TP", "U_Tp_V"), ("F_FACTOR", "F_factor"))
               if len(rf_eff) and rf_eff[c].isna().all()]
    if not_set:
        print("ℹ %s: Bedeutung der Abweichung für die Bewertung nicht bestimmt – weder in Abschnitt 1 gesetzt noch in der Ergebnisdatei gespeichert: %s"
              % (n, ", ".join(not_set)))
    lvc, lvc_eff = earthing.check_lv_network(e, response)
    REDUCTION[n] = dict(reduction_factor=rf, reduction_factor_eff=rf_eff, lv_network_check=lvc,
                        lv_network_check_eff=lvc_eff)
    if len(rf_eff):
        display(Markdown("**Reduktionsfaktor** (effektiv und je Harmonische)")); show(rf_eff); show(rf)
        interactive.reduction_factor(rf, rf_eff).show()
    if len(lvc):
        display(Markdown("**Ortsnetz ↔ Erdungsmessung**")); show(lvc_eff); show(lvc)
        interactive.lv_network(e.table("station_lv_network"), lvc).show()
    for h in earthing.reduction_hints(rf_eff, lvc):
        print("⚠ %s: %s" % (n, h))

## 6 · Berichtstabellen und Abbildungen exportieren

Berichtstabellen (Spannungen vor/nach, Erdschlussreststrom, Aufteilung 50 Hz, Erdungsspannungen, Schirmstrom, Trafo,
Bewertungsgrößen, Ersatzquellen) als Excel-Mappe, Word-Datei, CSV und Markdown; Abbildungen in Endgröße ohne Titel als
PDF/SVG/PNG.

In [ ]:
slug = report.slug                  # Namen in Datei- und Tabellennamen: Kleinbuchstaben, Buchstaben/Ziffern/_ (wie in den Berichtstabellen)
OUT = CAMPAIGN_FOLDER / LAYOUT.results / ("campaign" if LAYOUT.language == "en" else "Kampagne")
FIG = CAMPAIGN_FOLDER / LAYOUT.figures / ("campaign" if LAYOUT.language == "en" else "Kampagne")
TABLES = report.report_tables(RESULTS, series=MAIN, Z_E=Z_E, comparison=OTHER)
TABLES["overview"] = campaign.overview(RESULTS)
for n, e in STATIONS.items():                     # Station: frequenzselektiv und Verhältnis zur Erdungsmessung
    TABLES["station_frequency_" + slug(n)] = e.table("station_frequency")
    if n in COMPARISONS and len(COMPARISONS[n][1]):
        TABLES["earthing_ratio_" + slug(n)] = COMPARISONS[n][1]
    if not e.tables.get("station_lv_network_eff", pd.DataFrame()).empty:
        TABLES["station_lv_network_eff_" + slug(n)] = e.table("station_lv_network_eff")
        TABLES["station_lv_network_" + slug(n)] = e.table("station_lv_network")
    for t_, df in REDUCTION.get(n, {}).items():
        if len(df):
            TABLES[t_ + "_" + slug(n)] = df
if PQ is not None:                                 # PQ-Messung: Quantile, Kennwerte, Versuche, Reststrom an Z_E
    TABLES["pq_quantiles"] = PQ.table().query("order >= 2")
    TABLES["pq_summary"] = PQ.summary_table()
    for name_, df_ in (("pq_tests", PQ_COMPARISON), ("pq_scaling", PQ_SCALING)):
        if df_ is not None and len(df_):
            TABLES[name_] = df_
for name, df in TABLES.items():                    # Titel mit dem Namen der Messung wie angegeben
    display(Markdown("**%s**" % report.table_title(name, RESULTS))); display(report.format_table(df))
paths = report.export_tables(TABLES, str(OUT), prefix="campaign_", formats=("csv", "md"), names=RESULTS)
figures = []
for n, e in STATIONS.items():
    figures += report.save(report.station_frequency(e), str(FIG / ("station_frequency_" + slug(n))), EXPORT_FORMATS, close=True)
    figures += report.save(report.station_background(e), str(FIG / ("station_background_" + slug(n))), EXPORT_FORMATS, close=True)
    if n in COMPARISONS:
        try:
            figures += report.save(report.earthing_comparison(*COMPARISONS[n]),
                                   str(FIG / ("earthing_comparison_" + slug(n))), EXPORT_FORMATS, close=True)
        except ValueError as ex:
            print("ℹ %s: %s" % (n, ex))
    rd = REDUCTION.get(n, {})
    if len(rd.get("reduction_factor_eff", [])):
        figures += report.save(report.reduction_factor(rd["reduction_factor"], rd["reduction_factor_eff"]),
                               str(FIG / ("reduction_factor_" + slug(n))), EXPORT_FORMATS, close=True)
    if not e.tables.get("station_lv_network", pd.DataFrame()).empty:
        lvc = rd.get("lv_network_check")
        figures += report.save(report.lv_network(e.table("station_lv_network"), lvc if lvc is not None and len(lvc) else None),
                               str(FIG / ("lv_network_" + slug(n))), EXPORT_FORMATS, close=True)
sub_results = [RESULTS[m["name"]] for m in MEASUREMENTS if m["Z"] is not None and m["name"] in RESULTS]
if sub_results:
    f = report.spectrum_comparison(sub_results, [m["name"] for m in MEASUREMENTS if m["Z"] is not None and m["name"] in RESULTS])
    figures += report.save(f, str(FIG / "fault_current_spectrum_comparison"), EXPORT_FORMATS, close=True)
if MAIN is not None:                    # Ergebnis (SOURCE), die andere Quelle gestrichelt
    figures += report.save(report.source_fit(MAIN, Z_E=Z_E, comparison=OTHER), str(FIG / "source_fit"), EXPORT_FORMATS,
                           close=True)
    figures += report.save(report.source_plane(MAIN, MC), str(FIG / "source_impedance"), EXPORT_FORMATS, close=True)
    figures += report.save(report.ires_over_ze(MAIN, Z_E_MAX, LIMITS, IG_CHOICE, SCALING, comparison=OTHER),
                           str(FIG / "ires_over_ZE"), EXPORT_FORMATS, close=True)
if PQ is not None:
    figures += report.save(report.pq_series(PQ), str(FIG / "pq_series"), EXPORT_FORMATS, close=True)
    figures += report.save(report.pq_duration(PQ, levels={n: lv[PQ.dominant_order] for n, lv in (PQ_LEVELS or {}).items()
                                                          if PQ.dominant_order in lv}),
                           str(FIG / "pq_duration"), EXPORT_FORMATS, close=True)
    figures += report.save(report.pq_profile(PQ), str(FIG / "pq_profile"), EXPORT_FORMATS, close=True)
print("%d Tabellen-Dateien in %s" % (len(paths), OUT.relative_to(WORK_FOLDER)))
print("%d Abbildungsdateien in %s" % (len(figures), FIG.relative_to(WORK_FOLDER)))

## 7 · groundmeas: Messungen der Kampagne eintragen

Jede Messung aus `MEASUREMENTS` als Erdschlussmessung (`staged_fault_test`) in groundmeas – UW mit dem Fehlerstrom je
Ordnung und dem Fehlerwiderstand, Station zusätzlich mit Erdungs-/Berührungsspannung je Sonde und Frequenz
(fehlerbedingter Anteil) und Schirmstrom. Ort, Koordinaten, Anlagentyp, Spannungsebene und Entfernung zum UW stehen in
der Messorte-Datei der Kampagne (wird beim ersten Mal aus groundmeas vorbelegt angelegt – für **neue Orte Koordinaten
eintragen**). Zuerst nur Übersicht (Trockenlauf); geschrieben wird mit `GROUNDMEAS_EXPORT = True` – nach **einer**
Sicherungskopie der Datenbank; schon eingetragene Messungen werden erkannt und übersprungen (`"replace"` erneuert die
eigenen Einträge), andere Einträge werden nie verändert.

In [ ]:
legacy.check_option("GROUNDMEAS_EXPORT", GROUNDMEAS_EXPORT, (False, True, "replace"))
PLANS, BACKUP = {}, True
for m in MEASUREMENTS:
    if m["name"] not in RESULTS:
        continue
    try:
        PLANS[m["name"]] = p = earthing.export_to_groundmeas(
            RESULTS[m["name"]], CAMPAIGN_FOLDER, GROUNDMEAS_DB, location=m.get("location"), write=bool(GROUNDMEAS_EXPORT),
            existing="replace" if GROUNDMEAS_EXPORT == "replace" else "skip", uncertain_mode=GROUNDMEAS_UNCERTAIN,
            description=m.get("description", ""), timestamp=m.get("timestamp"), units=GROUNDMEAS_UNITS, backup=BACKUP)
        if p.get("written"):
            BACKUP = p["backup"]           # eine Sicherungskopie für alle Einträge dieses Laufs
    except FileNotFoundError as ex:
        print("ℹ groundmeas-Datenbank nicht verfügbar:", ex)
        break
    except ValueError as ex:
        print("ℹ %s: %s" % (m["name"], ex))
if PLANS:
    show(earthing.plan_overview(PLANS))
    for n, p in PLANS.items():
        for h in p["hints"]:
            print("ℹ %s: %s" % (n, h))
        for w in p["warnings"]:
            print("⚠ %s: %s" % (n, w))
    display(Markdown("**%s**" % Path(sites.path_for(CAMPAIGN_FOLDER)).relative_to(CAMPAIGN_FOLDER)))
    show(sites.load(CAMPAIGN_FOLDER).table())
    if isinstance(BACKUP, str):
        print("✓ eingetragen – Sicherungskopie: %s – jetzt GROUNDMEAS_EXPORT wieder auf False stellen." % BACKUP)
    elif any(p["writable"] for p in PLANS.values()):
        print("Trockenlauf – eintragen mit GROUNDMEAS_EXPORT = True (Werte je Messung: earthing.plan_items(PLANS[name])).")

## 8 · Bericht (Word)

Der Bericht der Kampagne als Word-Datei in `4_Bericht`: Titelseite, Kurzfassung, Inhalt, Kapitel 1 bis 3 (Aufbau und
Durchführung; Ergebnisse; Bewertung mit den Ersatzquellen, der Berührungsspannung an den Stationen, den Unsicherheiten
und der Zusammenfassung) und Anhänge (Messdaten, Plausibilitätsprüfung, Formelzeichen). Jede Zahl stammt aus den
Objekten dieses Notebooks und den Ergebnisdateien, Verläufe und Fehlerzeitpunkte aus den Rohdaten. Fotos vor Ort
(`PHOTOS`, Dateinamen im Medienordner) werden eingesetzt, fehlende sind gerahmte Platzhalter; die Empfehlung bleibt ein
markierter Platzhalter, bis `REPORT["recommendation"]` sie angibt. Mit `REPORT_TEMPLATE` übernimmt der Bericht
Formatvorlagen, Kopf- und Fußzeile und Logo der eigenen Organisation. Die Datei in Word öffnen und die Felder
aktualisieren (Inhaltsverzeichnis, Nummern der Abbildungen und Tabellen). Erneutes Ausführen ersetzt einen unveränderten
Bericht; ein seitdem in Word gespeicherter bleibt als `<Name>_<Datum>_<Uhrzeit>.docx` erhalten.

In [ ]:
REPORT_FILE = None
if REPORT is None:
    print("ℹ Kein Bericht – REPORT ist None (Abschnitt 1).")
elif not document.docx_available():
    print("ℹ Kein Bericht – python-docx fehlt: pip install 'groundfault[export]'")
elif not RESULTS:
    print("ℹ Kein Bericht – keine Ergebnisse (Abschnitt 2).")
else:
    REPORT_OUT = document.campaign_report(
        RESULTS, info=REPORT, measurements=MEASUREMENTS, campaign_folder=str(CAMPAIGN_FOLDER), series=MAIN,
        comparison=OTHER, monte_carlo=MC, loo=LOO if MAIN is not None else None, Z_E=Z_E, IG=IG_CHOICE, limits=LIMITS,
        u_tp=U_TP, f_factor=F_FACTOR, uncertainty_I=UNCERTAINTY_I, uncertainty_U=UNCERTAINTY_U, scaling=SCALING, pq=PQ,
        pq_levels=PQ_LEVELS, pq_comparison=PQ_COMPARISON, earthing=COMPARISONS, reduction=REDUCTION, photos=PHOTOS,
        template=REPORT_TEMPLATE, styles=REPORT_STYLES, language=LANGUAGE, pdf=REPORT_PDF)
    REPORT_FILE = Path(REPORT_OUT["path"])
    print("✓ Bericht: %s%s" % (REPORT_FILE.relative_to(WORK_FOLDER), " + PDF" if REPORT_OUT["pdf"] else ""))
    print(i18n.format_text("  Bewertung mit {} · {} Abbildungen", REPORT_OUT["basis"],
                           len(REPORT_OUT["figures"])))
    for p_ in REPORT_OUT["placeholders"]:     # noch in Word zu füllen
        print("  [ ]", p_)
    for h in REPORT_OUT["hints"]:
        print("ℹ", h)